# Realized Profit Exit Signal

**Hypothesis:** Just like Realized Loss spikes mark bottoms, Realized Profit spikes should mark tops.

**The Logic:**
- MVRV > 2.5 = unrealized gains exist (STATE)
- Realized Profit spike = gains being TAKEN (ACTION)
- Combined = actual distribution/top

**Mirror of Entry Signal:**
- Entry: SOPR < 1 + Realized Loss Z > 0.5
- Exit: SOPR > 1.0x + Realized Profit Z > X?

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

print("Realized Profit Exit Signal Analysis 📊")

In [ ]:
# Load data
DATA_DIR = Path("../data/raw")

price = pd.read_parquet(DATA_DIR / "price.parquet").rename(columns={"value": "price"}).set_index("time")
sopr = pd.read_parquet(DATA_DIR / "sopr.parquet").rename(columns={"value": "sopr"}).set_index("time")
sopr_sth = pd.read_parquet(DATA_DIR / "sopr_sth.parquet").rename(columns={"value": "sopr_sth"}).set_index("time")
mvrv = pd.read_parquet(DATA_DIR / "mvrv.parquet").rename(columns={"value": "mvrv"}).set_index("time")
realized_loss = pd.read_parquet(DATA_DIR / "realized_loss.parquet").rename(columns={"value": "realized_loss"}).set_index("time")

# Check if realized_profit exists
rp_path = DATA_DIR / "realized_profit.parquet"
if rp_path.exists():
    realized_profit = pd.read_parquet(rp_path).rename(columns={"value": "realized_profit"}).set_index("time")
    print("✅ Realized Profit data found!")
else:
    realized_profit = None
    print("❌ Realized Profit not found - need to fetch from Glassnode")

# Join available data
df = price.join(sopr, how='inner').join(sopr_sth, how='inner').join(mvrv, how='inner').join(realized_loss, how='inner')
if realized_profit is not None:
    df = df.join(realized_profit, how='inner')

df = df.sort_index()
df = df[df.index >= '2020-01-01'].dropna()

print(f"\nData: {len(df)} rows ({df.index.min().date()} to {df.index.max().date()})")
print(f"Columns: {list(df.columns)}")

In [ ]:
# If we don't have realized_profit, we can approximate with SOPR-based metrics
# SOPR > 1 means selling at profit, higher = more profit being realized

# Calculate z-scores
df['rl_z30'] = (df['realized_loss'] - df['realized_loss'].rolling(30).mean()) / df['realized_loss'].rolling(30).std()

if 'realized_profit' in df.columns:
    df['rp_z30'] = (df['realized_profit'] - df['realized_profit'].rolling(30).mean()) / df['realized_profit'].rolling(30).std()
    df['rp_z14'] = (df['realized_profit'] - df['realized_profit'].rolling(14).mean()) / df['realized_profit'].rolling(14).std()
    df['rp_z7'] = (df['realized_profit'] - df['realized_profit'].rolling(7).mean()) / df['realized_profit'].rolling(7).std()

# SOPR-based profit proxy
df['sopr_excess'] = df['sopr'] - 1  # How much above breakeven
df['sopr_excess_z14'] = (df['sopr_excess'] - df['sopr_excess'].rolling(14).mean()) / df['sopr_excess'].rolling(14).std()
df['sopr_excess_z30'] = (df['sopr_excess'] - df['sopr_excess'].rolling(30).mean()) / df['sopr_excess'].rolling(30).std()

df['sth_excess'] = df['sopr_sth'] - 1
df['sth_excess_z14'] = (df['sth_excess'] - df['sth_excess'].rolling(14).mean()) / df['sth_excess'].rolling(14).std()

# Forward returns
df['fwd_30d'] = df['price'].shift(-30) / df['price'] - 1
df['fwd_90d'] = df['price'].shift(-90) / df['price'] - 1

df = df.dropna()
print(f"Data after calculations: {len(df)} rows")

---
## 1. Known Tops - What Were Realized Metrics?

In [ ]:
# Major tops
major_tops = [
    ('2021-04-14', 'April 2021 ATH'),
    ('2021-11-10', 'Nov 2021 ATH'),
    ('2024-03-14', 'March 2024 ATH'),
    ('2024-12-17', 'Dec 2024 ATH'),
    ('2021-05-10', 'Pre-China Crash'),
    ('2022-03-28', 'Bear Rally 1'),
    ('2022-11-05', 'Pre-FTX'),
]

print("REALIZED METRICS AT MAJOR TOPS")
print("="*120)

cols = ['SOPR', 'STH-SOPR', 'MVRV', 'SOPR Excess Z14']
if 'realized_profit' in df.columns:
    cols.append('RP Z30')

print(f"{'Date':<12} {'Event':<20} {'Price':>10} {'SOPR':>8} {'STH':>8} {'MVRV':>8} {'SOPR Ex Z':>10} {'30d Fwd':>10}")
print("-"*100)

top_data = []
for date_str, name in major_tops:
    try:
        target = pd.Timestamp(date_str, tz='UTC')
        idx = df.index.get_indexer([target], method='nearest')[0]
        row = df.iloc[idx]
        actual_date = df.index[idx]
        
        top_data.append({
            'date': actual_date,
            'name': name,
            'price': row['price'],
            'sopr': row['sopr'],
            'sth': row['sopr_sth'],
            'mvrv': row['mvrv'],
            'sopr_ex_z': row['sopr_excess_z14'],
            'fwd_30d': row['fwd_30d'],
        })
        
        fwd = f"{row['fwd_30d']*100:+.0f}%" if pd.notna(row['fwd_30d']) else 'N/A'
        print(f"{actual_date.strftime('%Y-%m-%d'):<12} {name:<20} ${row['price']:>9,.0f} {row['sopr']:>8.3f} {row['sopr_sth']:>8.3f} {row['mvrv']:>8.2f} {row['sopr_excess_z14']:>10.2f} {fwd:>10}")
    except Exception as e:
        print(f"Error: {e}")

tops_df = pd.DataFrame(top_data)

In [ ]:
# Summary
print("\nSUMMARY AT TOPS")
print("="*60)
print(f"Average SOPR: {tops_df['sopr'].mean():.3f}")
print(f"Average STH-SOPR: {tops_df['sth'].mean():.3f}")
print(f"Average MVRV: {tops_df['mvrv'].mean():.2f}")
print(f"Average SOPR Excess Z14: {tops_df['sopr_ex_z'].mean():.2f}")
print(f"")
print(f"% where SOPR > 1.02: {(tops_df['sopr'] > 1.02).mean()*100:.0f}%")
print(f"% where SOPR > 1.05: {(tops_df['sopr'] > 1.05).mean()*100:.0f}%")
print(f"% where SOPR Excess Z > 1: {(tops_df['sopr_ex_z'] > 1).mean()*100:.0f}%")
print(f"% where SOPR Excess Z > 1.5: {(tops_df['sopr_ex_z'] > 1.5).mean()*100:.0f}%")

---
## 2. Forward Returns by Exit Signal

In [ ]:
print("FORWARD RETURNS BY EXIT SIGNAL")
print("="*100)
print(f"{'Signal':<40} {'Avg 30d':>12} {'Avg 90d':>12} {'Days':>10} {'% Time':>10}")
print("-"*100)

exit_signals = [
    ('All days', df['sopr'] > 0),
    
    # Unrealized (MVRV) - for comparison
    ('MVRV > 2.0', df['mvrv'] > 2.0),
    ('MVRV > 2.5', df['mvrv'] > 2.5),
    
    # Realized (SOPR level)
    ('SOPR > 1.02', df['sopr'] > 1.02),
    ('SOPR > 1.05', df['sopr'] > 1.05),
    ('SOPR > 1.08', df['sopr'] > 1.08),
    
    # Realized (SOPR z-score - spike detection)
    ('SOPR Excess Z14 > 1.0', df['sopr_excess_z14'] > 1.0),
    ('SOPR Excess Z14 > 1.5', df['sopr_excess_z14'] > 1.5),
    ('SOPR Excess Z14 > 2.0', df['sopr_excess_z14'] > 2.0),
    ('SOPR Excess Z30 > 1.0', df['sopr_excess_z30'] > 1.0),
    ('SOPR Excess Z30 > 1.5', df['sopr_excess_z30'] > 1.5),
    
    # STH-SOPR (short-term holders taking profit)
    ('STH-SOPR > 1.02', df['sopr_sth'] > 1.02),
    ('STH-SOPR > 1.05', df['sopr_sth'] > 1.05),
    ('STH Excess Z14 > 1.0', df['sth_excess_z14'] > 1.0),
    ('STH Excess Z14 > 1.5', df['sth_excess_z14'] > 1.5),
    
    # Combined: Unrealized + Realized
    ('MVRV>2 + SOPR>1.02', (df['mvrv'] > 2) & (df['sopr'] > 1.02)),
    ('MVRV>2 + SOPR Z>1', (df['mvrv'] > 2) & (df['sopr_excess_z14'] > 1)),
    ('MVRV>2.5 + SOPR>1.05', (df['mvrv'] > 2.5) & (df['sopr'] > 1.05)),
    ('MVRV>2.5 + SOPR Z>1.5', (df['mvrv'] > 2.5) & (df['sopr_excess_z14'] > 1.5)),
]

results = []
for name, cond in exit_signals:
    subset = df[cond]
    if len(subset) > 10:
        avg_30 = subset['fwd_30d'].mean() * 100
        avg_90 = subset['fwd_90d'].mean() * 100
        pct = len(subset) / len(df) * 100
        results.append({'name': name, 'avg_30': avg_30, 'avg_90': avg_90, 'days': len(subset), 'pct': pct})
        print(f"{name:<40} {avg_30:>+11.1f}% {avg_90:>+11.1f}% {len(subset):>10} {pct:>9.1f}%")

# Sort by most negative 90d return (best exit signal)
print("\n" + "="*60)
print("BEST EXIT SIGNALS (most negative forward returns)")
print("="*60)
sorted_results = sorted(results, key=lambda x: x['avg_90'])
for i, r in enumerate(sorted_results[:10]):
    print(f"{i+1}. {r['name']}: {r['avg_90']:+.1f}% (90d), {r['pct']:.1f}% of time")

---
## 3. Compare Entry vs Exit Signals

In [ ]:
print("ENTRY vs EXIT SIGNAL COMPARISON")
print("="*80)

# Entry signals (should have POSITIVE forward returns)
entry_signals = [
    ('SOPR < 1 (entry)', df['sopr'] < 1, 'positive'),
    ('STH-SOPR < 1 (entry)', df['sopr_sth'] < 1, 'positive'),
    ('SOPR < 1 + RL Z > 0.5', (df['sopr'] < 1) & (df['rl_z30'] > 0.5), 'positive'),
]

# Exit signals (should have NEGATIVE forward returns)
exit_sigs = [
    ('SOPR > 1.05 (exit)', df['sopr'] > 1.05, 'negative'),
    ('SOPR Z14 > 1.5 (exit)', df['sopr_excess_z14'] > 1.5, 'negative'),
    ('MVRV>2.5 + SOPR Z>1.5', (df['mvrv'] > 2.5) & (df['sopr_excess_z14'] > 1.5), 'negative'),
]

print(f"\n{'Signal':<35} {'Type':<10} {'Avg 90d':>12} {'Correct?':>10}")
print("-"*70)

for name, cond, expected in entry_signals + exit_sigs:
    subset = df[cond]
    if len(subset) > 10:
        avg_90 = subset['fwd_90d'].mean() * 100
        correct = (avg_90 > 0 and expected == 'positive') or (avg_90 < 0 and expected == 'negative')
        correct_str = '✅' if correct else '❌'
        print(f"{name:<35} {expected:<10} {avg_90:>+11.1f}% {correct_str:>10}")

---
## 4. Visualize Distribution Events

In [ ]:
# Plot price with SOPR z-score
fig = make_subplots(
    rows=3, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.05,
    row_heights=[0.5, 0.25, 0.25],
    subplot_titles=('BTC Price', 'SOPR', 'SOPR Excess Z-Score (14d)')
)

# Price
fig.add_trace(go.Scatter(x=df.index, y=df['price'], name='Price', line=dict(color='orange')), row=1, col=1)

# Mark tops
for _, t in tops_df.iterrows():
    fig.add_trace(go.Scatter(
        x=[t['date']], y=[t['price']],
        mode='markers', marker=dict(size=12, color='red', symbol='triangle-down'),
        showlegend=False, hovertext=t['name']
    ), row=1, col=1)

# SOPR
fig.add_trace(go.Scatter(x=df.index, y=df['sopr'], name='SOPR', line=dict(color='blue')), row=2, col=1)
fig.add_hline(y=1, line_dash='dash', line_color='gray', row=2, col=1)
fig.add_hline(y=1.05, line_dash='dash', line_color='red', row=2, col=1)

# SOPR Z-Score
fig.add_trace(go.Scatter(x=df.index, y=df['sopr_excess_z14'], name='SOPR Z', line=dict(color='purple')), row=3, col=1)
fig.add_hline(y=0, line_dash='dash', line_color='gray', row=3, col=1)
fig.add_hline(y=1.5, line_dash='dash', line_color='red', row=3, col=1)
fig.add_hline(y=-1.5, line_dash='dash', line_color='green', row=3, col=1)

fig.update_yaxes(type='log', row=1, col=1)
fig.update_layout(height=800, title='Price vs SOPR (Realized Profit-Taking Detection)', showlegend=False)
fig.show()

---
## 5. Backtest Realized Exit Strategy

In [ ]:
from numba import njit

@njit
def exit_realized_profit(price_arr, sopr_arr, sopr_z_arr, mvrv_arr, entry_idx,
                         sopr_exit=1.05, sopr_z_exit=1.5, mvrv_context=2.0,
                         trail_pct=0.20, stop_loss=0.25):
    """
    Exit when:
    1. MVRV > context (market expensive) AND
    2. SOPR > threshold OR SOPR Z > threshold (profit-taking happening)
    Then activate trailing stop
    """
    entry_price = price_arr[entry_idx]
    peak = entry_price
    trail_active = False
    
    for j in range(entry_idx + 1, len(price_arr)):
        price = price_arr[j]
        sopr = sopr_arr[j]
        sopr_z = sopr_z_arr[j]
        mvrv = mvrv_arr[j]
        
        if price > peak:
            peak = price
        
        # Activate trail when: expensive market + profit-taking
        if not trail_active:
            if mvrv > mvrv_context and (sopr > sopr_exit or sopr_z > sopr_z_exit):
                trail_active = True
        
        # Exit conditions
        if trail_active and price <= peak * (1 - trail_pct):
            return j, price, 'trail'
        
        if (price - entry_price) / entry_price <= -stop_loss:
            return j, price, 'stop'
    
    return len(price_arr) - 1, price_arr[-1], 'hold'


@njit
def exit_simple_trail(price_arr, sopr_arr, sopr_z_arr, mvrv_arr, entry_idx,
                      trail_pct=0.30, stop_loss=0.25, **kwargs):
    """Baseline: Simple trailing stop"""
    entry_price = price_arr[entry_idx]
    peak = entry_price
    
    for j in range(entry_idx + 1, len(price_arr)):
        price = price_arr[j]
        if price > peak:
            peak = price
        if price <= peak * (1 - trail_pct):
            return j, price, 'trail'
        if (price - entry_price) / entry_price <= -stop_loss:
            return j, price, 'stop'
    return len(price_arr) - 1, price_arr[-1], 'hold'

In [ ]:
def run_backtest(df, entries, exit_func, initial_capital=100000, **kwargs):
    """Run backtest with given exit function"""
    price_arr = df['price'].values
    sopr_arr = df['sopr'].values
    sopr_z_arr = df['sopr_excess_z14'].values
    mvrv_arr = df['mvrv'].values
    dates = df.index
    entry_indices = np.where(entries.values)[0]
    
    trades = []
    i = 0
    
    while i < len(entry_indices):
        entry_idx = entry_indices[i]
        exit_idx, exit_price, exit_reason = exit_func(price_arr, sopr_arr, sopr_z_arr, mvrv_arr, entry_idx, **kwargs)
        
        entry_price = price_arr[entry_idx]
        net_return = (exit_price / entry_price) - 1 - 0.002
        
        trades.append({
            'entry_date': dates[entry_idx],
            'exit_date': dates[exit_idx],
            'entry_price': entry_price,
            'exit_price': exit_price,
            'exit_reason': exit_reason,
            'net_return': net_return,
            'days_held': (dates[exit_idx] - dates[entry_idx]).days
        })
        
        while i < len(entry_indices) and entry_indices[i] <= exit_idx:
            i += 1
    
    trades_df = pd.DataFrame(trades)
    if len(trades_df) > 0:
        equity = [initial_capital]
        for _, t in trades_df.iterrows():
            equity.append(equity[-1] * (1 + t['net_return']))
        trades_df['equity'] = equity[1:]
    return trades_df


def calc_metrics(trades, initial_capital=100000):
    if len(trades) == 0:
        return None
    final = trades['equity'].iloc[-1]
    total_ret = (final / initial_capital) - 1
    years = (trades['exit_date'].iloc[-1] - trades['entry_date'].iloc[0]).days / 365.25
    win_rate = (trades['net_return'] > 0).mean()
    returns = trades['net_return'].values
    sharpe = (returns.mean() / returns.std()) * np.sqrt(len(trades)/years) if returns.std() > 0 and years > 0 else 0
    
    equity = [initial_capital] + list(trades['equity'])
    peak, max_dd = equity[0], 0
    for eq in equity:
        if eq > peak: peak = eq
        dd = (eq - peak) / peak
        if dd < max_dd: max_dd = dd
    
    return {
        'total_return': total_ret,
        'sharpe': sharpe,
        'max_dd': max_dd,
        'win_rate': win_rate,
        'n_trades': len(trades),
        'avg_hold': trades['days_held'].mean()
    }

In [ ]:
# Entry signal
entry_cond = df['sopr_sth'] < 1
entries = entry_cond & ~entry_cond.shift(1).fillna(False)

print("REALIZED PROFIT EXIT STRATEGY TEST")
print("Entry: STH-SOPR < 1")
print("="*120)

strategies = [
    # Baselines
    ('Simple 30% Trail', exit_simple_trail, {'trail_pct': 0.30}),
    ('Simple 20% Trail', exit_simple_trail, {'trail_pct': 0.20}),
    
    # Realized profit exits
    ('MVRV>2 + SOPR>1.05 → 20% trail', exit_realized_profit,
     {'mvrv_context': 2.0, 'sopr_exit': 1.05, 'sopr_z_exit': 99, 'trail_pct': 0.20}),
    ('MVRV>2 + SOPR Z>1.5 → 20% trail', exit_realized_profit,
     {'mvrv_context': 2.0, 'sopr_exit': 99, 'sopr_z_exit': 1.5, 'trail_pct': 0.20}),
    ('MVRV>2.5 + SOPR>1.05 → 15% trail', exit_realized_profit,
     {'mvrv_context': 2.5, 'sopr_exit': 1.05, 'sopr_z_exit': 99, 'trail_pct': 0.15}),
    ('MVRV>2.5 + SOPR Z>1.5 → 15% trail', exit_realized_profit,
     {'mvrv_context': 2.5, 'sopr_exit': 99, 'sopr_z_exit': 1.5, 'trail_pct': 0.15}),
    ('MVRV>2 + (SOPR>1.05 OR Z>1.5) → 20%', exit_realized_profit,
     {'mvrv_context': 2.0, 'sopr_exit': 1.05, 'sopr_z_exit': 1.5, 'trail_pct': 0.20}),
]

print(f"{'Strategy':<45} {'Return':>10} {'Sharpe':>8} {'MaxDD':>8} {'Win%':>7} {'Trades':>7} {'AvgHold':>8}")
print("-"*100)

all_results = []
for name, func, kwargs in strategies:
    trades = run_backtest(df, entries, func, **kwargs)
    m = calc_metrics(trades)
    if m:
        print(f"{name:<45} {m['total_return']*100:>+9.0f}% {m['sharpe']:>8.2f} {m['max_dd']*100:>7.0f}% {m['win_rate']*100:>6.0f}% {m['n_trades']:>7} {m['avg_hold']:>7.0f}d")
        all_results.append({'name': name, 'metrics': m, 'trades': trades})

In [ ]:
# Trade log for best realized exit strategy
best = max([r for r in all_results if 'MVRV' in r['name']], key=lambda x: x['metrics']['total_return'])

print(f"\nBEST REALIZED EXIT: {best['name']}")
print("="*100)
print(best['trades'][['entry_date', 'exit_date', 'entry_price', 'exit_price', 'exit_reason', 'net_return', 'days_held']].to_string())

---
## 6. Summary

In [ ]:
print("\n" + "="*70)
print("SUMMARY: REALIZED vs UNREALIZED EXIT SIGNALS")
print("="*70)

print("""
📊 THE HYPOTHESIS:
   Entry: SOPR < 1 works because it's REALIZED (action)
   Exit: MVRV > 2.5 fails because it's UNREALIZED (state)
   
   Solution: Use REALIZED profit-taking for exits!
   - SOPR > 1.05 = selling at 5%+ profit
   - SOPR Z > 1.5 = unusually high profit-taking

📈 RESULTS:
""")

# Compare simple vs realized
simple = [r for r in all_results if 'Simple' in r['name']][0]
realized = max([r for r in all_results if 'MVRV' in r['name']], key=lambda x: x['metrics']['total_return'])

print(f"   Simple Trail: {simple['metrics']['total_return']*100:+,.0f}%")
print(f"   Realized Exit: {realized['metrics']['total_return']*100:+,.0f}%")
print(f"   Winner: {'Realized Exit ✅' if realized['metrics']['total_return'] > simple['metrics']['total_return'] else 'Simple Trail ✅'}")

print("""
💡 KEY INSIGHT:
   - Unrealized metrics (MVRV) tell you market is EXPENSIVE
   - Realized metrics (SOPR spike) tell you people ARE SELLING
   - Combined = "expensive + distribution" = better timing
""")